# RAG - Retrieval Augmented Generations    
* Is a patern that can improve the effecacy of a LLM application by leveraging custom data   
* Is done by retieving data/documents relevants to a question or task and providing them as context to augment the prompts to a LLM to improve generation.

#### RAG Use Cases    

* Q&A Chatbots   
* Search Augmentation   
* Content Criation and Summarization

#### Main concepts of RAG Workflow   
**1.  Index and Embed**: An embedding model used to creating vector representation of the documents and users queries.   
**2.  Vector store**: Specialized to store unstructured data indexed by vectros. Vectors can be sotred with a **vector DB**, **library** or **plugin**   
**3.  Retrieval**: Search stored vectors uing similairity search to efficiently retrieve relavant information   
**4.  Filtering & Reranking**: The process of selectting or raking retrieved documents before passing as context. Filtering can be **pre-, in-, post-query**    
**5.  Prompt Augmentation**: Prompt engineering workflow to enhace context via injections of data retrieved from Vector store    
**6.  Generation**: A LLM used for generating a response for the user's request. 

#### Benefits of RAG Architecture
* Up-to-date and accurate response    
* Reducing inaccurate response or hallucination    
* Damin-specific contextitualization   
* Efficiency and cost0effectiveness

### Using MLflow for RAG applications

**Main concepts**
* Open soruce platform for machine learning lifecycle
* Co-developed by Databricks and ML community
* Pre-installedt on the Databricks Runtime for ML
* Operationallizing Generative AI development lifecycle

**Make your GenAI workflow more manageble and transparent**
* Record LLM parameters such as temperature and model cinfigurations
* Log metrics and compare them to get insights about the performance and accuracy LLMs.
* Store and manage outputs arifacts such as visualization images and serielized models.
* Store models's source code from the run


**A standard format for packeginf machine learning models**
* Each MLflow Model is a directory containing arbritrary files, together with MLmodel file.
* MLModel file can be defin multiple flavors that the model can be viewed in
* With MLflow models deployments tools can understand the model
* Model lifecycle can contain additional metadata such as signature, input example, etc.

**Built-in model flavors** Python Function (mlflow.pyfunc)
* Servers as a default model interface for MLFlow Python models
* Any MLFlow Python model is expected to be loadable as a python funtions
* Allows you to deploy models as pythons function
* It includdes all the information necessary to load anduse a model
* Some functions: **log_model, save_model, load_model, predict**

**Model Registry**
A centralized model store   
* Deploy and organize
* Model versioning
* Model lifecycle management with aliases. Example: champion for the production stage
* Collaboration and permission management
* Full model lineage
* Tagging and annotations.


### How to evaluating RAG Pipeline
* Due to RAG complexity, when evaluating RAG solutions, we need to evaluate each component separetely and together
* Components to evaluate
  * Chunckin: method, size
  * Embedding model
  * Vector store
    * Retrieval and re-ranker
  * Generator

![](/Workspace/Users/eliel.paes@digiage.com.br/GenAILearning/Databricks_GenAIEngineering/5_RAG/img/RAGEvaluation.png)

#### Context Precision
* Signal-to-noise ratio for the retrieved context
* Based on query and context(s)
* It assesses whether the chuncks/nodes in the retrieval context ranked higher than irrelevants one.


#### Context Relevancy
* Measure the relevancy of the retrieved context
* Based on both the query and context(s)
* It does not necessarily consider the factual accuracy but focuses on the well the answer addresees the posed question

#### Context Recall
* Measure the extent to which all relevant entities and information are retrieved and mentioned in the context provided
* Base on Ground Thruth and retrieved context(s)

#### Faithfulness
* Measures the factual accuracy of the generated asnswer in relation to the provided context
* Based on the response and retrieved context(s)

#### Answer Relevancy
Generation related metrics

* Assesses how pertinent and applicable the generated response is to the user's inital query.
* Based on the alignment of the response with the user's intent or query specifics.


#### Answer Correctness
Generation related metrics

* Measures the accureacy of the generated answer when compared to the ground truth
* Based on the ground truth and the response
* Encompasses both semantic and factual similarity with the ground truth.


### MLflow (LLM) Evaluation
Efficiently evaluates retriaveers and LLMs    

**Batch comparison:**    
Compare fundational models with fine-tuned modles on many questios    

**Rapid and scalable experimentation:**    
MLflow can eavluate unstructured outpus automatically, rapidly, and at low-cost.   

**Cost-Effetive:**    
Automating evaluation swith LLMs, can seve time on huma evaluation.   
PS: Human evaluation is the state of the art and should not be ignored.




In [0]:
%pip install mlflow==2.10.1 lxml==4.9.3 langchain==0.1.5 databricks-vectorsearch==0.22 cloudpickle==2.2.1 databricks-sdk==0.18.0 cloudpickle==2.2.1 pydantic==2.5.2 openpyxl
%pip install "mlflow[genai]"
%pip install pip mlflow[databricks]==2.10.1

dbutils.library.restartPython()

In [0]:
import os
import io
import re
import json
import numpy as np
import pandas as pd
import langchain
import mlflow

from databricks.vector_search.client import VectorSearchClient
from langchain.vectorstores import DatabricksVectorSearch
from langchain.embeddings import DatabricksEmbeddings
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatDatabricks
from langchain.prompts import PromptTemplate
from mlflow.deployments import set_deployments_target
from mlflow.models import infer_signature

from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

In [0]:
catalog = "analytics"
db_name = "bronze"
table_name = f"pdf_text_embeddings"
embedding_endpoint = "databricks-bge-large-en"
chat_endpoint = "databricks-llama-4-maverick"
index_name = f"{table_name}_vsc"
vsc_endpoint = "eliel_paes_studies"
vs_index_fullname = f"{catalog}.{db_name}.{index_name}"
chat_history = []

In [0]:
chat_model = ChatDatabricks(endpoint = chat_endpoint,
                            temperature = 0,
                            max_tokens = 8096) 

In [0]:
# defining the embedding model
embedding_model = DatabricksEmbeddings(endpoint = "databricks-bge-large-en")

# defining the retriever method
def get_retriever():
    """
    
    
    """

    # Getting the vector search index
    vsc = VectorSearchClient()
    vs_index = vsc.get_index(endpoint_name = vsc_endpoint, index_name = vs_index_fullname)


    # creating a retriever
    vectorstore = DatabricksVectorSearch(vs_index, text_column = "content", embedding = embedding_model)

    return vectorstore.as_retriever(search_kwargs = {"k": 5})

In [0]:
TEMPLATE = """You are a assistant for reading papers and answering questions about them. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Use the following pieces of context to answer the question at the end. 

<context>
{context}
</context>

Question: {question}

Answer:
"""

prompt = PromptTemplate(template = TEMPLATE, input_variables = ["context", "question"])

chain = RetrievalQA.from_chain_type(llm = chat_model,
                                    chain_type = "stuff",
                                    retriever = get_retriever(),
                                    chain_type_kwargs = {"prompt": prompt})                 

In [0]:
question = {"query":  "What is the three-layered approach for auditing LLMs proposed in this article? "}
answer = chain.invoke(question)
print(answer["result"])

# Evaluating the RAG Application

In [0]:
# quesions with the ground truth
data = json.load(open("../config/questions.json"))

# creating a dataframe with the question and the grond_truth answers to be used in the evaluation
evaluation_df = spark.createDataFrame(data).select(F.col("question"), F.col("answer").alias("ground_truth"))

In [0]:
@pandas_udf("array<string>")
def get_contexts(question: pd.Series) -> pd.Series:
    """
    
    """
    # breking the pandas series into batches
    max_batch_size = 100
    idx = [i for i in range(0, len(question), max_batch_size)]
    batches = [question.iloc[i:i+max_batch_size] for i in idx]

    # getting the context for each batch
    contexts = []
    for batch in batches:
        contexts.extend(batch.apply(lambda x: [c.page_content for c in get_retriever().invoke(x)]))

    return pd.Series(contexts)

@pandas_udf("string")
def get_answers(question: pd.Series) -> pd.Series:
    """
    
    """
    # breking the pandas series into batches
    max_batch_size = 100
    idx = [i for i in range(0, len(question), max_batch_size)]
    batches = [question.iloc[i:i+max_batch_size] for i in idx]

    # getting the rag answer for each batch
    contexts = []
    for batch in batches:
        contexts.extend(batch.apply(lambda x: chain.invoke(x)["result"]))

    return pd.Series(contexts)

In [0]:
evaluation_df = evaluation_df.withColumn("retrieved_context", get_contexts(F.col("question")))\
                            .withColumn("rag_answer", get_answers(F.col("question")))

display(evaluation_df)

In [0]:
set_deployments_target("databricks")

gtp_oss_answer_similarity = mlflow.metrics.genai.answer_similarity(
    model = f"endpoints:/databricks-gpt-oss-20b"        
)

gtp_oss_relevance = mlflow.metrics.genai.relevance(
    model = f"endpoints:/databricks-gpt-oss-20b"    
)

results =  mlflow.evaluate(
    data = evaluation_df.toPandas(), # dataframe with the questions and the ground truth answers
    evaluator_config={
                        "col_mapping": {
                        "inputs": "question",          # your dataframe column → expected "inputs"
                        "context": "retrieved_context", # optional, but useful                        
                        }
                    },    
    model_type = "question-answering",
    targets = "ground_truth",
    predictions = "rag_answer", # column with the rag answers
    extra_metrics = [gtp_oss_answer_similarity, gtp_oss_relevance], # extra metrics to be calculated,
    evaluators = "default"
    )

display(results.tables['eval_results_table'])

In [0]:
# set model registry to UC
mlflow.set_registry_uri("databricks-uc")
model_name = f"{catalog}.{db_name}.rag_app_papers"

with mlflow.start_run(run_name = "rag_app_papers") as run:
    signature = infer_signature(evaluation_df.toPandas()["question"], evaluation_df.toPandas()["rag_answer"])
    model_info = mlflow.langchain.log_model(
        chain, 
        loader_fn = get_retriever, 
        artifact_path = "papers_chain",
        registered_model_name = model_name,
        input_example = evaluation_df.toPandas().loc[0, "question"],
        pip_requirements = [
            "mlflow==" + mlflow.__version__,
            "langchain==" + langchain.__version__,
            "databricks-vectorsearch"
        ],        
        signature = signature
    )

#### 